In [ ]:
#1st task: Basic multiprocessing with Queues

import multiprocessing
import time
import random

def worker(task_queue, results_queue):
    """Function run by worker processes."""
    while True:
        # Get a task from the queue
        task = task_queue.get()
        
        # Check for the 'Poison Pill' (None) to shut down
        if task is None:
            break
        
        try:
            # Simulate CPU-intensive work
            time.sleep(random.uniform(0.1, 0.4)) 
            result = f"DONE: {task} (by {multiprocessing.current_process().name})"
            results_queue.put(result)
        except Exception as e:
            results_queue.put(f"ERROR on {task}: {e}")

if __name__ == "__main__":
    # Initialize Queues
    task_queue = multiprocessing.Queue()
    results_queue = multiprocessing.Queue()
    
    num_tasks = 20
    num_workers = multiprocessing.cpu_count()
    
    # 1. Start Worker Processes
    processes = []
    for i in range(num_workers):
        p = multiprocessing.Process(target=worker, args=(task_queue, results_queue))
        p.start()
        processes.append(p)

    # 2. Producer: Add tasks to the queue
    for i in range(num_tasks):
        task_queue.put(f"Job_{i}")

    # 3. Add Poison Pills (one for every worker started)
    for _ in range(num_workers):
        task_queue.put(None)

    # 4. Collect Results (Do this BEFORE or DURING joining to avoid deadlock)
    results = []
    for _ in range(num_tasks):
        results.append(results_queue.get())

    # 5. Clean up processes
    for p in processes:
        p.join()
    
    # Print the output
    print(f"--- All {len(results)} tasks completed ---")
    for r in sorted(results): # Sorted just to see them in order
        print(r)

In [ ]:
#2nd task: Shared State with multiprocessing.Manager

import multiprocessing

def update_shared_stats(worker_id, shared_dict, lock):
    # Simulating work and updating a shared "Dashboard"
    for i in range(100):
        with lock: # Critical: Prevent Race Conditions on the shared dict
            shared_dict['total_iterations'] += 1
            shared_dict['last_worker'] = worker_id

if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        # This dict exists in a server process and is synced across all forks
        dashboard = manager.dict({'total_iterations': 0, 'last_worker': None})
        lock = manager.Lock()
        
        pool = []
        for i in range(4):
            p = multiprocessing.Process(target=update_shared_stats, args=(i, dashboard, lock))
            pool.append(p)
            p.start()

        for p in pool: p.join()

        print(f"Final Dashboard State: {dashboard}")
        print(f"Total Iterations: {dashboard['total_iterations']}, Last Worker: {dashboard['last_worker']}")
       

In [ ]:
#3rd task Shared Memory with multiprocessing.Array

import multiprocessing
import ctypes
import numpy as np

# Global variable for the worker processes to access
shared_array_base = None

def init_worker(shared_array_ptr):
    """Initializes the global variable in each child process."""
    global shared_array_base
    shared_array_base = shared_array_ptr

def compute_on_shared(idx, shape):
    # Access existing memory buffer without copying
    # get_obj() provides access to the underlying synchronizable C object
    existing_shmem = np.frombuffer(shared_array_base.get_obj(), dtype=ctypes.c_double)
    arr = existing_shmem.reshape(shape)
    
    # Write directly to shared memory
    arr[idx] = np.sin(idx)

if __name__ == "__main__":
    size = 1000000
    shape = (size,)
    
    # 1. Allocate raw shared memory (Synchronized Array)
    # 'd' is the typecode for double
    shared_memory = multiprocessing.Array(ctypes.c_double, size)
    
    # 2. Use a Pool to manage worker processes
    # We pass the shared memory to the initializer so every worker has it
    with multiprocessing.Pool(processes=multiprocessing.cpu_count(), 
                             initializer=init_worker, 
                             initargs=(shared_memory,)) as pool:
        
        # 3. Map the work (indices 0 to 999999) to the compute function
        # We pass 'shape' as a repeated argument
        indices = range(size)
        pool.starmap(compute_on_shared, [(i, shape) for i in range(100)]) # Testing first 100

    # 4. Verify results in the main process
    final_arr = np.frombuffer(shared_memory.get_obj(), dtype=ctypes.c_double)
    print(f"Index 50 value: {final_arr[50]}")
    print(f"Expected value: {np.sin(50)}")

In [ ]:
#4th task: Synchronization with multiprocessing.Barrier 

import multiprocessing
import time
import random

def stage_worker(barrier, worker_id):
    # Simulate work for Stage 1
    time.sleep(random.uniform(0.5, 1.5))
    print(f"Worker {worker_id} finished Stage 1")
    
    try:
        # All processes pause here until the count hits the defined number (4)
        barrier.wait() 
    except multiprocessing.BrokenBarrierError:
        print(f"Worker {worker_id} exiting due to barrier break.")
        return

    print(f"Worker {worker_id} starting Stage 2")

if __name__ == "__main__":
    num_workers = 4
    
    # 1. Create the Barrier for 4 parties
    sync_point = multiprocessing.Barrier(num_workers)
    
    # 2. Create and start the processes
    processes = []
    for i in range(num_workers):
        p = multiprocessing.Process(target=stage_worker, args=(sync_point, i))
        p.start()
        processes.append(p)
    
    # 3. Ensure all processes finish before the main script ends
    for p in processes:
        p.join()

    print("All workers have completed both stages.")

In [ ]:
#5th task: Timeout and Termination with multiprocessing.Process

import multiprocessing
import time

def heavy_task():
    """A task that might take too long."""
    print("Worker: Starting a very long job...")
    time.sleep(60)  # Simulating a 1-minute task
    print("Worker: Finished!")

if __name__ == "__main__":
    p = multiprocessing.Process(target=heavy_task)
    p.start()

    # 1. Wait for up to 10 seconds
    p.join(timeout=10)

    # 2. Check if it's still running after the timeout
    if p.is_alive():
        print("Task exceeded 10s. Terminating...")
        p.terminate()  # Sends a SIGTERM
        
        # 3. Best Practice: join again to clean up the zombie process
        p.join() 
        print("Process safely cleaned up.")
    else:
        print("Task finished on time.")